In [21]:
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import ElasticNet
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [22]:
TRAIN_data = r"C:\Users\hp\Desktop\Data_analysis-and-ML\train.csv"
TEST_data = r"C:\Users\hp\Desktop\Data_analysis-and-ML\test.csv"
SUBMISSION_data = r"C:\Users\hp\Desktop\Data_analysis-and-ML\sample_submission.csv"

train = pd.read_csv(TRAIN_data)
test = pd.read_csv(TEST_data)

print(train.shape, test.shape)
train.head()

(1241, 81) (219, 80)


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,985,90,RL,75.0,10125,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,8,2009,COD,Normal,125969
1,778,20,RL,100.0,13350,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,MnPrv,NaN,0,6,2006,WD,Normal,142555
2,708,120,RL,48.0,6240,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2009,WD,Normal,254214
3,599,20,RL,80.0,12984,Pave,NaN,Reg,Bnk,AllPub,...,0,NaN,NaN,NaN,0,3,2006,WD,Normal,217309
4,875,50,RM,52.0,5720,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,8,2009,WD,Abnorml,66406


In [23]:
train = train.drop_duplicates()
test_ID = test["Id"]

def features(df):
    df = df.copy()

    if "Id" in df.columns:
        df = df.drop(columns=["Id"])

    na_as_none_cols = [
        "Alley", "BsmtQual", "BsmtCond", "BsmtExposure",
        "BsmtFinType1", "BsmtFinType2", "FireplaceQu",
        "GarageType", "GarageFinish", "GarageQual", "GarageCond",
        "PoolQC", "Fence", "MiscFeature"
    ]

    for col in na_as_none_cols:
        if col in df.columns:
            df[col] = df[col].fillna("None")

    zero_fill_cols = [
        "GarageYrBlt", "GarageArea", "GarageCars",
        "BsmtFinSF1", "BsmtFinSF2", "BsmtUnfSF", "TotalBsmtSF",
        "BsmtFullBath", "BsmtHalfBath", "MasVnrArea"
    ]

    for col in zero_fill_cols:
        if col in df.columns:
            df[col] = df[col].fillna(0)
    
    if {"1stFlrSF", "2ndFlrSF", "TotalBsmtSF"}.issubset(df.columns):
        df["TotalSF"] = df["1stFlrSF"] + df["2ndFlrSF"] + df["TotalBsmtSF"]

    if {"FullBath", "HalfBath", "BsmtFullBath", "BsmtHalfBath"}.issubset(df.columns):
        df["TotalBath"] = (
            df["FullBath"]
            + 0.5 * df["HalfBath"]
            + df["BsmtFullBath"]
            + 0.5 * df["BsmtHalfBath"]
        )

    if {"YrSold", "YearBuilt"}.issubset(df.columns):
        df["HouseAge"] = df["YrSold"] - df["YearBuilt"]

    if {"YrSold", "YearRemodAdd"}.issubset(df.columns):
        df["RemodAge"] = df["YrSold"] - df["YearRemodAdd"]

    if {"GrLivArea", "LotArea"}.issubset(df.columns):
        df["LivLotRatio"] = df["GrLivArea"] / (df["LotArea"] + 1)

    if {"OverallQual", "OverallCond"}.issubset(df.columns):
        df["TotalHomeQuality"] = df["OverallQual"] * df["OverallCond"]

    if {"OpenPorchSF", "EnclosedPorch", "3SsnPorch", "ScreenPorch"}.issubset(df.columns):
        df["TotalPorchSF"] = (
            df["OpenPorchSF"]
            + df["EnclosedPorch"]
            + df["3SsnPorch"]
            + df["ScreenPorch"]
        )

    if "GarageArea" in df.columns:
        df["HasGarage"] = (df["GarageArea"] > 0).astype(int)

    if "TotalBsmtSF" in df.columns:
        df["HasBsmt"] = (df["TotalBsmtSF"] > 0).astype(int)

    if "Fireplaces" in df.columns:
        df["HasFireplace"] = (df["Fireplaces"] > 0).astype(int)

    if "PoolArea" in df.columns:
        df["HasPool"] = (df["PoolArea"] > 0).astype(int)

    if "2ndFlrSF" in df.columns:
        df["Has2ndFloor"] = (df["2ndFlrSF"] > 0).astype(int)

    return df

train_fe = features(train)
test_fe = features(test)

X = train_fe.drop(columns=["SalePrice"])
y = train_fe["SalePrice"]
X_test = test_fe.copy()

cat_cols = X.select_dtypes(include=["object"]).columns.tolist()
num_cols = X.select_dtypes(exclude=["object"]).columns.tolist()

In [24]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, num_cols),
    ("cat", categorical_pipeline, cat_cols)
])

In [25]:
model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", ElasticNet(alpha=0.0005, l1_ratio=0.1, max_iter=20000, random_state=42))
])

def cv_mae_original_scale(model, X, y, n_splits=5):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    scores = []

    for fold, (tr_idx, va_idx) in enumerate(kf.split(X), start=1):
        X_train, X_valid = X.iloc[tr_idx], X.iloc[va_idx]
        y_train, y_valid = y.iloc[tr_idx], y.iloc[va_idx]

        m = clone(model)
        m.fit(X_train, np.log1p(y_train))

        preds = np.expm1(m.predict(X_valid))
        preds = np.clip(preds, 10000, None)

        mae = mean_absolute_error(y_valid, preds)
        scores.append(mae)
        print(f"Fold {fold}: {mae:,.2f}")

    print("Mean CV MAE:", f"{np.mean(scores):,.2f}")
    print("Std CV MAE: ", f"{np.std(scores):,.2f}")
    return scores

scores = cv_mae_original_scale(model, X, y)


Fold 1: 16,325.80
Fold 2: 16,451.03
Fold 3: 21,797.91
Fold 4: 14,874.99
Fold 5: 16,772.16
Mean CV MAE: 17,244.38
Std CV MAE:  2,368.33


In [26]:
model.fit(X, np.log1p(y))
test_preds = np.expm1(model.predict(X_test))
test_preds = np.clip(test_preds, 10000, None)

submission = pd.DataFrame({
    "Id": test_ID,
    "SalePrice": test_preds
})

submission.to_csv("submission_ML1.csv", index=False)
print("submission.csv created")

submission.csv created
